In [ ]:
%load_ext autoreload
%autoreload 2

import os
import yaml
import json
import polars as pl
import pandas as pd
from tqdm import tqdm

from plotnine import *
import matplotlib.pyplot as plt

from anngeno import AnnGeno
from scripts import get_burdens
import multiprocessing

num_cores = multiprocessing.cpu_count()
print(num_cores)

## Get missense variants for required gene

In [ ]:
pg = pl.read_parquet('/home/dnanexus/data_dir/250717_proteingym_SNP_DMS_scores_human_coding_genes.parquet').filter(pl.col('gene_name').is_in(['BRCA1']))
pg

In [ ]:
pg['file_name'].value_counts().sort('count', descending=True)

In [ ]:
plt.hist(pg['dms_score'], bins=100)
plt.show()

In [ ]:
anno = pl.read_parquet('/home/dnanexus/data_dir/genebass_1e6_coding_variants.ag/annotations.parquet').with_columns(
    pl.when(
        pl.col('amino_acids').is_not_null() & pl.col('protein_position').is_not_null()
    ).then(
        pl.col('amino_acids').str.split('/').list.get(0) +
        pl.col('protein_position').str.split('/').list.get(0) +
        pl.col('amino_acids').str.split('/').list.get(1)
    ).otherwise(None).alias('mutant')
).filter(
    # Filter for BRCA1 gene
    pl.col('region').is_in(['ENSG00000012048'])
)

id_cols = ['chrom', 'pos', 'ref', 'alt', 'id', 'region', 'col', 'AF_ukb', 'mutant', 'amino_acids', 'protein_position', 'consequence']
missense_annos = ['loftee_hc', 'CADD_RAW', 'am_pathogenicity', 'Consequence_missense_variant', 'PolyPhen', 'CADD_SIFTval', 'CADD_priPhCons', 'CADD_mamPhCons', 'CADD_verPhCons', 'gpn_score']

anno = anno.select(id_cols + missense_annos)
anno

In [ ]:
anno.filter(pl.col('Consequence_missense_variant')==1)

In [ ]:
pg.filter(~pl.col('mutant').is_in(anno['mutant']))

In [ ]:
brca_df = pg.filter(pl.col('file_name').is_in(['BRCA1_HUMAN_Findlay_2018']))[['mutant', 'dms_score', 'gene_name', 'file_name', 'region']].join(anno.filter(pl.col('region') == 'ENSG00000012048'), on=['region', 'mutant'], how='inner')

annos2compare = missense_annos + ['dms_score']
brca_df

### Check how exp. scores correlate with comp. scores

In [ ]:
(
    ggplot(brca_df, aes(x='dms_score', y='am_pathogenicity')) +
    geom_point() +
    geom_smooth(method='lm', se=True, color='darkred') +
    labs(x='DMS Score', y='Alphamissense') +
    theme_bw()
)

In [ ]:
(
    ggplot(brca_df, aes(x='dms_score', y='CADD_RAW')) +
    geom_point() +
    geom_smooth(method='lm', se=True, color='darkred') +
    labs(x='DMS Score', y='CADD RAW') +
    theme_bw()
)

In [ ]:
(
    ggplot(brca_df, aes(x='dms_score', y='gpn_score')) +
    geom_point() +
    geom_smooth(method='lm', se=True, color='darkred') +
    labs(x='DMS Score', y='GPN-MSA') +
    theme_bw()
)

### Pos-neg split scores

In [ ]:

def check_positive_negative(df: pl.DataFrame, columns):
    results = {}
    for col in columns:
        if col in df.columns:
            non_null = df.select(pl.col(col).drop_nulls())[col]
            if non_null.is_empty():
                results[col] = False  # Only nulls
            else:
                min_val = non_null.min()
                max_val = non_null.max()
                results[col] = (min_val < 0) and (max_val > 0)
        else:
            results[col] = False  # Column not found
    return results

# Example usage:
positive_negative_check = check_positive_negative(brca_df, annos2compare)

# Print the results
for column, has_both in positive_negative_check.items():
    if has_both:
        print(f"Column '{column}': Contains both positive and negative values.")

In [ ]:
def split_pos_neg_lazy(df: pl.LazyFrame, columns):
    # Start with the lazy frame
    lf = df

    for col in columns:
        if col in df.columns:
            pos_col = (
                pl.when(pl.col(col) > 0)
                .then(pl.col(col))
                .otherwise(0)
                .alias(f"{col}_pos")
            )

            neg_col = (
                pl.when(pl.col(col) < 0)
                .then(pl.col(col))
                .otherwise(0)
                .alias(f"{col}_neg")
            )

            lf = lf.with_columns([pos_col, neg_col])

    return lf

# Get the columns that contain both positive and negative values
mix_cols = [k for k, v in positive_negative_check.items() if v]

# Split the positive and negative values into separate columns
split_ann = split_pos_neg_lazy(brca_df.lazy(), mix_cols).collect()
split_ann

## Compute burdens

In [ ]:
split_ann

In [ ]:
%%time

config_path = "/home/dnanexus/ukbgym/config_dms.yaml"
output_dir = "/home/dnanexus/data_dir/dms_burdens/"
gene_id = 'ENSG00000012048'

get_burdens.compute_and_store_burdens(
    config_path=config_path,
    gene_list=[gene_id],
    output_dir=output_dir,
    only_snps=True,
    na_mask=True,
    overwrite=True,
    gene_chunk_size=1,
    sample_chunk_size=50_000,
    new_annotation_df=split_ann.lazy(),
    variant_subset=split_ann['id'].unique().to_list()
)

In [ ]:
b = pl.read_parquet(f'{output_dir}/{gene_id}.parquet')
b.drop_nans()

## get phenotypes

In [ ]:
phenos = pl.read_parquet('/home/dnanexus/data_dir/phenotypes/phenotypes190_missing80_unique2.parquet').select(['individual', 'jurgens_breast_cancer']).drop_nulls()
phenos['jurgens_breast_cancer'].value_counts()

In [ ]:
prs = pl.read_parquet('/home/dnanexus/data_dir/phenotypes/PRS190_missing80_unique2.parquet').select(['individual', 'jurgens_breast_cancer_prs']).drop_nulls()
plt.hist(prs['jurgens_breast_cancer_prs'], bins=100)
plt.show()

In [ ]:
config_path = f'/home/dnanexus/ukbgym/config_dms.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

cov_list = config.get("covariates")

cov_df = pl.read_parquet('/home/dnanexus/data_dir/phenotypes/250629_quant_phenotypes_covariates_genetic_pcs_prs_corrected.parquet').rename({'eid':'individual'}).select(['individual'] + cov_list).with_columns(pl.col('individual').cast(pl.Int64).alias('individual'))
cov_df

### correct for PRS and covariates

In [ ]:
all_df = phenos.join(prs, on='individual', how='inner').join(cov_df, on='individual', how='inner')
all_pd = all_df.to_pandas()
all_pd

In [ ]:
# Restrict to EUR ancestry
eur_samples = pl.read_csv('/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv')
eur_samples

all_pd = all_pd[all_pd['individual'].isin(eur_samples['eid'].to_list())]
all_pd

In [ ]:
import statsmodels.api as sm

pheno = 'jurgens_breast_cancer'

combined_df = pd.DataFrame(index=all_pd.index)

y = all_pd[pheno]
X = all_pd.drop(columns=[pheno])
X = sm.add_constant(X)  # Add a constant term for the intercept

# Fit the model
model = sm.OLS(y, X).fit()

# Save residuals
residuals = pd.Series(model.resid, index=combined_df.index, name=f'{pheno}_residual')

pdf = pl.DataFrame(pd.concat([all_pd[['individual', 'jurgens_breast_cancer', 'jurgens_breast_cancer_prs']], residuals], axis=1)).with_columns(pl.col('individual').cast(pl.String).alias('individual'))
pdf

## Merge and Plot

In [ ]:
gene_id = 'ENSG00000012048'
gis = pl.read_parquet(f'/home/dnanexus/data_dir/dms_burdens/{gene_id}.parquet').rename({'sample_id':'individual'})
gis

In [ ]:
plt_df = gis.join(pdf, on='individual', how='inner').to_pandas()
plt_df

In [ ]:
plt_df.loc[plt_df['annotation'] == 'dms_score_pos', ['max', 'top2', 'sum']] *= -1
plt_df

In [ ]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

def get_roc_pr_results(plt_df, phenotype_col, score_agg='max'):
    """
    Computes ROC and PR curves + AUC and auPRC for each annotation.
    Cleans NaNs and ensures PR curves start at real precision.
    """
    roc_list = []
    pr_list = []
    summary_list = []

    annotations = plt_df["annotation"].unique()

    for anno in annotations:
        sub = plt_df[plt_df["annotation"] == anno]
        y_true = sub[phenotype_col]
        y_score = sub[score_agg]

        # Binarize residuals
        y_bin = (y_true > 0).astype(int)

        # Drop NaNs in both
        mask = ~(y_bin.isna() | y_score.isna())
        y_bin_clean = y_bin[mask]
        y_score_clean = y_score[mask]

        # Flip sign if needed: higher = more likely positive
        # y_score_clean = -y_score_clean

        if len(y_bin_clean.unique()) < 2:
            print(f"Skipping '{anno}': only one class present.")
            continue

        # === ROC ===
        fpr, tpr, _ = roc_curve(y_bin_clean, y_score_clean)
        roc_auc = auc(fpr, tpr)

        roc_df = pd.DataFrame({
            "fpr": fpr,
            "tpr": tpr,
            "curve": "ROC",
            "annotation": anno
        })
        roc_list.append(roc_df)

        # === PR ===
        precision, recall, _ = precision_recall_curve(y_bin_clean, y_score_clean)
        auprc = average_precision_score(y_bin_clean, y_score_clean)

        # `precision_recall_curve` is fine: first recall is 0, first precision is real.
        pr_df = pd.DataFrame({
            "recall": recall,
            "precision": precision,
            "curve": "PR",
            "annotation": anno
        })
        pr_list.append(pr_df)

        summary_list.append({
            "annotation": anno,
            "ROC_AUC": roc_auc,
            "auPRC": auprc
        })

    # Final tidy frames
    roc_data = pd.concat(roc_list, ignore_index=True)
    pr_data = pd.concat(pr_list, ignore_index=True)
    summary_df = pd.DataFrame(summary_list).sort_values(by='ROC_AUC', ascending=False)

    return roc_data, pr_data, summary_df

roc_data, pr_data, summary_df = get_roc_pr_results(plt_df, phenotype_col='jurgens_breast_cancer', score_agg='max')

summary_df

In [ ]:
# Merge AUROC values into roc_data
roc_plot = roc_data.merge(summary_df[['annotation', 'ROC_AUC']], on='annotation', how='left')

# Format new label column
roc_plot['annotation_label'] = roc_plot.apply(
    lambda row: f"{row['annotation']} (AUC={row['ROC_AUC']:.2f})",
    axis=1
)

roc_plot['linetype'] = roc_plot['annotation'].apply(lambda x: 'solid' if 'dms' in x else 'dashed')
roc_plot

In [ ]:
plot_anno = ['am_pathogenicity','gpn_score_neg','dms_score_neg','dms_score_pos','PolyPhen', 'CADD_RAW_neg']

(
    ggplot(roc_plot[roc_plot['annotation'].isin(plot_anno)], aes(x="fpr", y="tpr", color="annotation_label"))
    + geom_line(aes(linetype="linetype"))
    + geom_abline(slope=1, intercept=0, color="grey")
    + scale_linetype_manual(values={"solid":"solid", "dashed":"dashed"})
    + labs(
        x="False Positive Rate",
        y="True Positive Rate",
        color="Annotation"
    )
    + theme_bw()
    + guides(linetype=False)
    + theme(
        figure_size=(8, 6),
        legend_title=element_text(size=10),
        legend_text=element_text(size=9),
        axis_title=element_text(size=12),
        axis_text=element_text(size=10)
    )
)
